# Group 8 — CIS 434 Scraper Notebook
**Humans of New York | Tumblr Re-Entry Strategy**

This notebook handles all data collection for the project:
1. **HONY Archive Scrape** — 2,000 posts from @humansofnewyork via Tumblr API v2
2. **Tag Benchmark Scrape** — ~2,000 posts from 6 narrative-adjacent Tumblr tags

Both CSVs are saved and reusable. Run this notebook once. The analysis notebook loads from these CSVs.

**Setup:** Set `TUMBLR_API_KEY` env var or paste your consumer key below.

In [ ]:
import os, time, html, re, csv
from datetime import datetime
import requests
import pandas as pd

API_KEY = os.environ.get("TUMBLR_API_KEY", "PASTE_YOUR_CONSUMER_KEY_HERE")
SLEEP   = 0.3   # seconds between API calls — stays well under rate limits

def strip_html(raw):
    if not raw: return ""
    text = re.sub(r"<[^>]+>", " ", raw)
    text = html.unescape(text)
    return re.sub(r"\s+", " ", text).strip()

def extract_text(post):
    t = post.get("type", "")
    if t == "text":   return strip_html(post.get("body","")) or strip_html(post.get("title",""))
    if t == "photo":  return strip_html(post.get("caption",""))
    if t == "quote":  return strip_html(post.get("text","")) + " " + strip_html(post.get("source",""))
    if t == "link":   return strip_html(post.get("description","")) or strip_html(post.get("title",""))
    if t == "video":  return strip_html(post.get("caption",""))
    if t == "answer": return f"Q: {strip_html(post.get('question',''))} A: {strip_html(post.get('answer',''))}"
    return strip_html(post.get("summary",""))

print("Helpers loaded.")

## 1. HONY Archive Scrape
Scrapes the @humansofnewyork Tumblr blog. Skips if CSV already exists.

In [ ]:
HONY_CSV    = "hony_posts_raw.csv"
HONY_TARGET = 2000
HONY_URL    = "https://api.tumblr.com/v2/blog/humansofnewyork/posts"

if os.path.exists(HONY_CSV):
    df_hony = pd.read_csv(HONY_CSV)
    print(f"Loaded existing {HONY_CSV}: {len(df_hony)} posts")
else:
    rows, offset = [], 0
    while len(rows) < HONY_TARGET:
        r = requests.get(HONY_URL, params={"api_key": API_KEY, "limit": 20, "offset": offset}, timeout=30)
        if r.status_code != 200:
            print(f"API error {r.status_code}"); break
        posts = r.json().get("response", {}).get("posts", [])
        if not posts: break
        for p in posts:
            rows.append({"post_id": p.get("id"), "post_url": p.get("post_url"),
                         "type": p.get("type"), "date": p.get("date"),
                         "timestamp": p.get("timestamp"),
                         "tags": "|".join(p.get("tags",[]) or []),
                         "note_count": p.get("note_count", 0),
                         "text": extract_text(p)})
        if offset % 200 == 0:
            print(f"  offset={offset}, collected={len(rows)}")
        offset += 20
        time.sleep(SLEEP)
    df_hony = pd.DataFrame(rows)
    df_hony.to_csv(HONY_CSV, index=False, quoting=csv.QUOTE_NONNUMERIC)
    print(f"Saved {len(df_hony)} posts → {HONY_CSV}")

print(f"Posts: {len(df_hony)} | Date range: {df_hony['date'].iloc[-1][:10]} to {df_hony['date'].iloc[0][:10]}")
df_hony.head(2)

## 2. Tag Benchmark Scrape
Scrapes 6 Tumblr tags to build a benchmark corpus for the tag recommender.
The `/v2/tagged` endpoint returns posts from all blogs using that tag.

**Tag selection rationale:** Tags chosen to surface long-form personal narrative content similar to HONY.
Broad creative writing tags (e.g. `#storytelling`) attract fan fiction and Sims 4 posts — we account for this
with a minimum word count filter (80+ words) when loading into the recommender.

In [ ]:
BENCH_CSV    = "tag_benchmark_posts.csv"
BENCH_TARGET = 400   # per tag
BENCH_URL    = "https://api.tumblr.com/v2/tagged"

BENCH_TAGS = [
    "storytelling",
    "portrait photography",
    "personal narrative",
    "new york city",
    "family stories",
    "human interest",
]

def scrape_tag(tag, target=BENCH_TARGET):
    rows, before_ts = [], None
    while len(rows) < target:
        params = {"api_key": API_KEY, "tag": tag, "limit": 20}
        if before_ts:
            params["before"] = before_ts
        r = requests.get(BENCH_URL, params=params, timeout=30)
        if r.status_code != 200:
            print(f"  API error {r.status_code}"); break
        posts = r.json().get("response", [])
        if not posts: break
        for p in posts:
            rows.append({"post_id": p.get("id"), "post_url": p.get("post_url"),
                         "type": p.get("type"), "date": p.get("date"),
                         "timestamp": p.get("timestamp"),
                         "tags": "|".join(p.get("tags",[]) or []),
                         "note_count": p.get("note_count", 0),
                         "text": extract_text(p),
                         "query_tag": tag})
        before_ts = posts[-1].get("timestamp")
        time.sleep(SLEEP)
    return rows[:target]

if os.path.exists(BENCH_CSV):
    df_bench = pd.read_csv(BENCH_CSV)
    print(f"Loaded existing {BENCH_CSV}: {len(df_bench)} posts")
else:
    all_rows = []
    for tag in BENCH_TAGS:
        print(f"Scraping #{tag}...")
        rows = scrape_tag(tag)
        all_rows.extend(rows)
        print(f"  collected {len(rows)} posts (total so far: {len(all_rows)})")
    df_bench = pd.DataFrame(all_rows).drop_duplicates(subset="post_id").reset_index(drop=True)
    df_bench.to_csv(BENCH_CSV, index=False, quoting=csv.QUOTE_NONNUMERIC)
    print(f"Saved {len(df_bench)} posts → {BENCH_CSV}")

print(f"\nBenchmark posts by tag:")
print(df_bench['query_tag'].value_counts().to_string())
print(f"\nEngagement overview:")
print(df_bench.groupby('query_tag')['note_count'].agg(['median','mean']).round(1).to_string())

## 3. Data Quality Summary

In [ ]:
print("=" * 55)
print("HONY ARCHIVE")
print("=" * 55)
print(f"Posts scraped    : {len(df_hony)}")
print(f"Duplicates       : {df_hony['post_id'].duplicated().sum()}")
print(f"Missing text     : {df_hony['text'].isna().sum()}")
wc = df_hony['text'].fillna("").str.split().str.len()
print(f"Avg word count   : {wc.mean():.0f}")
print(f"Posts 100+ words : {(wc>=100).sum()}")
print(f"Note count median: {df_hony['note_count'].median():.0f}")

print()
print("=" * 55)
print("TAG BENCHMARK")
print("=" * 55)
print(f"Posts scraped    : {len(df_bench)}")
wc_b = df_bench['text'].fillna("").str.split().str.len()
print(f"Posts 80+ words  : {(wc_b>=80).sum()} (usable for recommender)")
print(f"Posts < 80 words : {(wc_b<80).sum()} (short captions, filtered out)")
print(f"Zero-note posts  : {(df_bench['note_count']==0).sum()} ({(df_bench['note_count']==0).mean()*100:.0f}%)")
print()
print("Note: The tag benchmark is used only for tag suggestions.")
print("Short posts are automatically filtered by the TagRecommender class.")